In [1]:
import os
import joblib
import pandas as pd
from trainingTools import getbest

In [2]:
ROUTE_DATA=r"C:\Users\Gabo\Downloads\models\models"
paths=os.listdir(ROUTE_DATA)
pathsPooling=[x for x in paths  if 'pooling' in x]
pathsBatch=[x for x in paths if 'seed' in x]
paths_level=[x for x in pathsBatch if 'level' in x]
paths_skill=[x for x in pathsBatch if 'skill' in x]
paths_subject=[x for x in pathsBatch if 'subject' in x]
paths_claridad=set(pathsBatch).difference(
    set(paths_level).union(paths_skill).union(paths_subject)
)

final_level=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_level])

final_subject=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_subject])

final_skill=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_skill])

final_claridad=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_claridad])

In [3]:
#parametros  para  obtener el pooling
groupcols=['pooling']
metrics= ['train_f1','val_f1','train_accuracy','val_accuracy']

pooling={}
for x in pathsPooling:
    db=pd.read_csv(f"{ROUTE_DATA}/{x}")
    cabezal=x.split('_')[1]
    best=getbest(db, groupcols,metrics)
    pooling[cabezal]=best['pooling']

print(pooling)

train_f1 :  0.8052916731328281
val_f1 :  0.7288924365263655
0.07639923660646264
{'claridad': 'mean', 'level': 'mean', 'skill': 'mean', 'subject': 'mean'}


In [4]:
groupcols=['num_hidden_layers',
       'hidden_dim', 'activation', 'normalization', 'dropout']

metrics= ['train_f1','val_f1','train_accuracy','val_accuracy']

a0=final_level[groupcols+metrics+['seed']]

a=a0[final_level['epoch']==final_level['best_epoch']]
b=a.groupby(groupcols)[metrics].agg('mean').reset_index()
print(a.shape)
print(b.shape)


(1296, 10)
(288, 9)


In [5]:
names=['level', 'skill','subject','claridad']
bases=[final_level,final_skill, final_subject, final_claridad]
info=dict(zip(names,bases))
params={}
for i,j in info.items():
    param=getbest(j,groupcols,metrics)
    print(i)
    print(param['train_f1'],param['val_f1'])
    pool=pooling[i]
    param['pooling']=pool
    params[i]=param
joblib.dump(params,'./finalCabezalParams.joblib')

level
0.9724864315861129 0.9300185014280814
train_f1 :  0.8998471242871351
val_f1 :  0.7079469703441164
0.19190015394301874
skill
0.8998471242871351 0.7079469703441164
subject
0.9981029466953156 0.9853864323226328
claridad
1.0 0.9985212792932189


['./finalCabezalParams.joblib']

In [6]:
params

{'level': {'num_hidden_layers': np.int64(1),
  'hidden_dim': np.int64(256),
  'activation': 'relu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.9724864315861129),
  'val_f1': np.float64(0.9300185014280814),
  'train_accuracy': np.float64(0.973775433308214),
  'val_accuracy': np.float64(0.933778715424285),
  'no_overfiting': np.True_,
  'pooling': 'mean'},
 'skill': {'num_hidden_layers': np.int64(3),
  'hidden_dim': np.int64(256),
  'activation': 'relu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.8998471242871351),
  'val_f1': np.float64(0.7079469703441164),
  'train_accuracy': np.float64(0.8960947412942337),
  'val_accuracy': np.float64(0.7021929824561403),
  'no_overfiting': np.False_,
  'pooling': 'mean'},
 'subject': {'num_hidden_layers': np.int64(3),
  'hidden_dim': np.int64(128),
  'activation': 'silu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.

In [7]:
a0=final_skill[groupcols+metrics+['seed']]

a=a0[final_skill['epoch']==final_skill['best_epoch']]
b=a.groupby(groupcols)[metrics].agg('mean').reset_index()
print(a.shape)
print(b.shape)


(1296, 10)
(288, 9)


In [14]:
a=final_skill.copy()
b=a[a['epoch']==a['best_epoch']]
b1=b.groupby(groupcols)[metrics+['train_loss','val_loss']].agg('mean').reset_index()
b1.sort_values(['train_accuracy'], ascending=False)


,num_hidden_layers,hidden_dim,activation,normalization,dropout,train_f1,val_f1,train_accuracy,val_accuracy,train_loss,val_loss
248,3,256,relu,batchnorm,0.0,0.899847,0.707947,0.896095,0.702193,0.240370,0.722511
272,3,512,relu,batchnorm,0.0,0.898031,0.711928,0.894497,0.704167,0.223041,0.743179
264,3,512,gelu,batchnorm,0.0,0.896507,0.708899,0.892664,0.700658,0.225066,0.747542
200,2,512,relu,batchnorm,0.0,0.895157,0.716416,0.891442,0.713816,0.257251,0.693220
192,2,512,gelu,batchnorm,0.0,0.895033,0.718656,0.891160,0.711184,0.245211,0.683836
...,...,...,...,...,...,...,...,...,...,...,...
42,0,256,silu,batchnorm,0.3,0.627702,0.616887,0.640350,0.632237,1.075501,1.086839
56,0,512,relu,batchnorm,0.0,0.623136,0.611195,0.639880,0.628728,1.071327,1.082865
63,0,512,relu,layernorm,0.5,0.624148,0.618192,0.639128,0.633772,1.065872,1.078596
69,0,512,silu,layernorm,0.1,0.625744,0.615592,0.639034,0.632456,1.070397,1.082798
